# MinerU on Dr.DocBench pages
Settings: **Accelerator = GPU**, **Internet = On**. Add the uploaded dataset as input. Use a *separate* notebook from PaddleOCR-VL (their dependencies conflict).
Run with `LIMIT = 20` first, check the printed output, then set `LIMIT = None`. MinerU's CLI has changed between versions, so the run cell tries the known command styles and prints what happened.

In [ ]:
!nvidia-smi | head -15
!python --version

In [ ]:
import os, glob, csv, zipfile, shutil, json, time, sys, subprocess
LIMIT = 20            # smoke test: run this many pages spread over dev+test. Set to None for the full run (1495 pages).
SPLITS = ("dev", "test")

manifests = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)      # Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/
zips = glob.glob("/kaggle/input/**/drdocbench_pages.zip", recursive=True)
if manifests:
    root = os.path.dirname(manifests[0])
elif zips:
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/data"); root = "/kaggle/working/data"
else:
    root = None
assert root, "dataset not found under /kaggle/input: " + str(glob.glob("/kaggle/input/**", recursive=False))
rows = [r for r in csv.DictReader(open(root + "/manifest.csv")) if r["split"] in SPLITS]
if LIMIT:
    step = max(1, len(rows) // LIMIT); rows = rows[::step][:LIMIT]
stem = lambda r: f"{r['doc']}_{r['page']}"
print(len(rows), "pages to process; first:", rows[0]["file"])

In [ ]:
# Pin the 2.x line: MinerU 4.x changed the CLI (needs a running server, no -p/-o/-b options); 2.x is what the benchmark README evaluated.
!python -m pip install -q -U "mineru[core]<3" || python -m pip install -q -U "mineru<3"
!pip show mineru | head -3
!mineru --version
!mineru --help 2>&1 | head -30

In [ ]:
# MinerU downloads its models on first use. Pre-fetching (optional; ignore errors, the first parse will fetch them):
!mineru-models-download --help 2>&1 | head -12

In [ ]:
os.environ["MINERU_MODEL_SOURCE"] = "huggingface"
OUT = "/kaggle/working/mineru_out"; os.makedirs(OUT, exist_ok=True)
todo = [r for r in rows if not os.path.exists(f"{OUT}/{stem(r)}/auto/{stem(r)}.md")]
print(len(todo), "pages left")
CHUNK = 5 if LIMIT else 40          # pages per mineru call: results are written only when a call finishes, so keep calls short
BUDGET = 9 * 3600                   # stop starting new chunks after this many seconds so the notebook can finish and zip cleanly
json.dump([{"file": os.path.abspath(f"{root}/{r['file']}"), "stem": stem(r)} for r in todo], open("/kaggle/working/rows.json", "w"))

WORKER = """
import os, sys, json, time, shutil, subprocess
shard, n, t_start, CHUNK, BUDGET = int(sys.argv[1]), int(sys.argv[2]), float(sys.argv[3]), int(sys.argv[4]), float(sys.argv[5])
os.environ["CUDA_VISIBLE_DEVICES"] = str(shard); os.environ["MINERU_MODEL_SOURCE"] = "huggingface"
OUT = "/kaggle/working/mineru_out"
items = json.load(open("/kaggle/working/rows.json"))[shard::n]
nchunks = -(-len(items) // CHUNK)
for i in range(0, len(items), CHUNK):
    if time.time() - t_start > BUDGET:
        print(f"shard {shard}: time budget reached, stopping", flush=True); break
    ch = items[i:i + CHUNK]; d = f"/kaggle/working/mineru_in_{shard}"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    for it in ch: os.symlink(it["file"], f"{d}/{it['stem']}.jpg")
    t0 = time.time()
    try:
        rc = subprocess.run(["mineru", "-p", d, "-o", OUT, "-b", "pipeline", "-l", "en"], capture_output=True, text=True, timeout=1500).returncode
    except subprocess.TimeoutExpired:
        rc = "timeout"
    done = sum(os.path.exists(f"{OUT}/{it['stem']}/auto/{it['stem']}.md") for it in ch)
    print(f"shard {shard}: chunk {i // CHUNK + 1}/{nchunks} rc={rc} {done}/{len(ch)} pages in {time.time() - t0:.0f}s", flush=True)
"""
open("/kaggle/working/mworker.py", "w").write(WORKER)

# warm-up on one page so the model download happens once, before the parallel workers start
d = "/kaggle/working/mineru_in_warm"; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
if todo: os.symlink(os.path.abspath(f"{root}/{todo[0]['file']}"), f"{d}/{stem(todo[0])}.jpg")
p = subprocess.run(["mineru", "-p", d, "-o", OUT, "-b", "pipeline", "-l", "en"], capture_output=True, text=True)
print("warm-up exit", p.returncode, (p.stdout + p.stderr)[-300:])

NGPU = 2; t_start = time.time()
procs = [subprocess.Popen([sys.executable, "/kaggle/working/mworker.py", str(k), str(NGPU), str(t_start), str(CHUNK), str(BUDGET)],
                          stdout=open(f"/kaggle/working/mineru{k}.log", "w"), stderr=subprocess.STDOUT) for k in range(NGPU)]
while any(q.poll() is None for q in procs):
    time.sleep(120)
    print(f"{time.time() - t_start:.0f}s: {len(glob.glob(OUT + '/*/auto/*.md'))} pages done |",
          " | ".join(open(f"/kaggle/working/mineru{k}.log").read().strip().split("\n")[-1][:90] for k in range(NGPU)), flush=True)
print("exit codes:", [q.returncode for q in procs])
for dd in glob.glob(OUT + "/*/auto/images"): shutil.rmtree(dd, ignore_errors=True)    # extracted figure crops are not needed
ok = len(glob.glob(OUT + "/*/auto/*.md")) > 0
print("SUCCESS" if ok else "no markdown produced - see /kaggle/working/mineru*.log", len(glob.glob(OUT + "/*/auto/*.md")), "pages")

In [ ]:
mds = glob.glob(OUT + "/**/*.md", recursive=True); print(len(mds), "markdown files")
allf = sorted(glob.glob(OUT + "/**/*", recursive=True)); print(len(allf), "files; sample:"); print("\n".join(f[len(OUT):] for f in allf[:25]))
for f in sorted(glob.glob(OUT + "/**/*middle*.json", recursive=True))[:1]:
    d = json.load(open(f)); print(f[len(OUT):], "keys:", list(d.keys())[:8])

In [ ]:
shutil.make_archive("/kaggle/working/mineru_results", "zip", OUT)
print("wrote /kaggle/working/mineru_results.zip", round(os.path.getsize("/kaggle/working/mineru_results.zip") / 1e6, 1), "MB -> download it from the Output tab")